1. Install and Import

In [2]:
!pip install pandas scikit-learn tensorflow

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 572.2/572.2 MB 58.0 MB/s  0:00:05m0:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.1/7.1 MB 192.0 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.6/4.6 MB 159.6 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.1/2.1 MB 132.3 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.5/24.5 MB 184.0 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13/13 [tensorflow]3 [tensorflow]


In [3]:
import tensorflow as tf
import numpy as np
import pandas as pd
import sklearn

from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import precision_recall_fscore_support, classification_report
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.layers import Input, Embedding, Conv1D, GlobalMaxPooling1D, Dense, Dropout, Concatenate
from tensorflow.keras.models import Model
from tensorflow.keras.callbacks import EarlyStopping

#Set random seed for reproducibility
RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)
tf.random.set_seed(RANDOM_SEED)

I0000 00:00:1787605244.643217   13342 cpu_feature_guard.cc:227] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.


2. Load pairwise gold dataset with bins

In [4]:
# Load the same dataset with fixed Bin assignments
df = pd.read_csv(
    "pairwise_gold_dataset_with_bins.csv",
    encoding="utf-8-sig"
)

# Check label distribution within each bin
print(pd.crosstab(df["Bin"], df["Label"]))

Label    0   1
Bin           
0      283  57
1      283  57
2      283  57
3      283  57
4      283  57
5      283  57
6      283  56
7      283  56
8      283  56
9      283  56


3. CNN input preparation

In [5]:
MAX_WORDS = 20000        # Maximum number of words in the tokenizer vocabulary
MAX_LEN = 64             # Maximum sequence length for each pair text
EMBEDDING_DIM = 128      # Dimension of trainable word embeddings

def make_pair_text(dataframe):
    # Combine two organization names into one input sequence
    return (
        dataframe["OrgName_1"].astype(str)
        + " [SEP] "
        + dataframe["OrgName_2"].astype(str)
    ).tolist()

4. CNN model

In [6]:
def build_cnn_model(vocab_size):
    # Define CNN architecture for pairwise text classification
    input_layer = Input(shape=(MAX_LEN,))

    # Convert token IDs into dense trainable embeddings
    x = Embedding(
        input_dim=vocab_size,
        output_dim=EMBEDDING_DIM,
        input_length=MAX_LEN
    )(input_layer)

    # Apply convolution filters with different kernel sizes
    conv3 = Conv1D(filters=128, kernel_size=3, activation="relu")(x)
    conv4 = Conv1D(filters=128, kernel_size=4, activation="relu")(x)
    conv5 = Conv1D(filters=128, kernel_size=5, activation="relu")(x)

    # Apply global max pooling to each convolution output
    pool3 = GlobalMaxPooling1D()(conv3)
    pool4 = GlobalMaxPooling1D()(conv4)
    pool5 = GlobalMaxPooling1D()(conv5)

    # Concatenate pooled features from different kernel sizes
    x = Concatenate()([pool3, pool4, pool5])

    # Add dropout to reduce overfitting
    x = Dropout(0.5)(x)

    # Add dense layer for classification
    x = Dense(128, activation="relu")(x)

    # Add another dropout layer
    x = Dropout(0.3)(x)

    # Output probability of match class
    output_layer = Dense(1, activation="sigmoid")(x)

    # Build model
    model = Model(inputs=input_layer, outputs=output_layer)

    # Compile model for binary classification
    model.compile(
        optimizer="adam",
        loss="binary_crossentropy",
        metrics=["accuracy"]
    )

    return model

5. 10-fold CNN evaluation

In [8]:
def run_cnn_10bin_evaluation(df):
    # Store fold-level evaluation metrics
    fold_results = []

    # Store pair-level predictions and probability scores
    pair_predictions = []

    for test_bin in range(10):
        # Use the next bin as development set
        dev_bin = (test_bin + 1) % 10

        # Use the remaining 8 bins as training set
        train_bins = [
            b for b in range(10)
            if b not in [test_bin, dev_bin]
        ]

        # Split data into train, development, and test sets
        train_df = df[df["Bin"].isin(train_bins)].reset_index(drop=True)
        dev_df = df[df["Bin"] == dev_bin].reset_index(drop=True)
        test_df = df[df["Bin"] == test_bin].reset_index(drop=True)

        # Print fold information
        print(f"\nFold {test_bin}")
        print("Train:", train_df.shape, train_df["Label"].value_counts().to_dict())
        print("Dev:", dev_df.shape, dev_df["Label"].value_counts().to_dict())
        print("Test:", test_df.shape, test_df["Label"].value_counts().to_dict())

        # Create text inputs for each split
        train_text = make_pair_text(train_df)
        dev_text = make_pair_text(dev_df)
        test_text = make_pair_text(test_df)

        # Extract labels
        y_train = train_df["Label"].values
        y_dev = dev_df["Label"].values
        y_test = test_df["Label"].values

        # Initialize tokenizer using training data only
        tokenizer = Tokenizer(
            num_words=MAX_WORDS,
            oov_token="[OOV]"
        )

        # Fit tokenizer only on training text to avoid data leakage
        tokenizer.fit_on_texts(train_text)

        # Convert text to padded token sequences
        X_train = pad_sequences(
            tokenizer.texts_to_sequences(train_text),
            maxlen=MAX_LEN,
            padding="post",
            truncating="post"
        )

        X_dev = pad_sequences(
            tokenizer.texts_to_sequences(dev_text),
            maxlen=MAX_LEN,
            padding="post",
            truncating="post"
        )

        X_test = pad_sequences(
            tokenizer.texts_to_sequences(test_text),
            maxlen=MAX_LEN,
            padding="post",
            truncating="post"
        )

        # Determine effective vocabulary size
        vocab_size = min(MAX_WORDS, len(tokenizer.word_index) + 1)

        # Build a new CNN model for each fold
        model = build_cnn_model(vocab_size)

        # Stop training when validation loss stops improving
        early_stopping = EarlyStopping(
            monitor="val_loss",
            patience=3,
            restore_best_weights=True
        )

        # Train CNN model
        model.fit(
            X_train,
            y_train,
            validation_data=(X_dev, y_dev),
            epochs=30,
            batch_size=32,
            callbacks=[early_stopping],
            verbose=1,
            class_weight={0: 1.0, 1: 5.0}
        )

        # Predict match probability for test pairs
        y_prob = model.predict(X_test).ravel()

        # Convert probability scores into binary predictions
        y_pred = (y_prob >= 0.5).astype(int)

        # Compute class-specific precision, recall, and F1
        class_precision, class_recall, class_f1, class_support = precision_recall_fscore_support(
            y_test,
            y_pred,
            labels=[0, 1],
            average=None,
            zero_division=0
        )

        # Save fold-level evaluation results
        fold_results.append({
            "Fold": test_bin,
            "Model": "CNN",

            "Match_Precision": class_precision[1],
            "Match_Recall": class_recall[1],
            "Match_F1": class_f1[1],
            "Match_Support": class_support[1],

            "NonMatch_Precision": class_precision[0],
            "NonMatch_Recall": class_recall[0],
            "NonMatch_F1": class_f1[0],
            "NonMatch_Support": class_support[0],
        })

        # Save pair-level predictions and probability scores
        temp_pred_df = test_df.copy()
        temp_pred_df["Fold"] = test_bin
        temp_pred_df["Model"] = "CNN"
        temp_pred_df["TrueLabel"] = y_test
        temp_pred_df["PredictedLabel"] = y_pred
        temp_pred_df["Probability_Match"] = y_prob
        temp_pred_df["Probability_NonMatch"] = 1 - y_prob

        pair_predictions.append(temp_pred_df)

        # Print classification report for this fold
        print(classification_report(
            y_test,
            y_pred,
            target_names=["NonMatch", "Match"],
            digits=4,
            zero_division=0
        ))

    # Combine fold-level results
    fold_results_df = pd.DataFrame(fold_results)

    # Combine pair-level prediction results
    pair_predictions_df = pd.concat(pair_predictions, ignore_index=True)

    return fold_results_df, pair_predictions_df

6. Run and save outputs

In [9]:
# Run CNN training and evaluation
cnn_fold_results_df, cnn_pair_predictions_df = run_cnn_10bin_evaluation(df)

# Save fold-level evaluation results
cnn_fold_results_df.to_csv(
    "cnn_fold_level_results_match_nonmatch.csv",
    index=False,
    encoding="utf-8-sig"
)

# Save pair-level prediction results
cnn_pair_predictions_df.to_csv(
    "cnn_pair_level_predictions_with_probability.csv",
    index=False,
    encoding="utf-8-sig"
)

# Summarize CNN performance across folds
cnn_summary_df = (
    cnn_fold_results_df
    .groupby("Model")[
        [
            "Match_Precision",
            "Match_Recall",
            "Match_F1",
            "NonMatch_Precision",
            "NonMatch_Recall",
            "NonMatch_F1"
        ]
    ]
    .agg(["mean", "std"])
)

cnn_summary_df.to_csv(
    "cnn_summary_match_nonmatch.csv",
    encoding="utf-8-sig"
)

print(cnn_summary_df)


Fold 0
Train: (2716, 9) {0: 2264, 1: 452}
Dev: (340, 9) {0: 283, 1: 57}
Test: (340, 9) {0: 283, 1: 57}
Epoch 1/30


/home/ec2-user/anaconda3/envs/pytorch/lib/python3.10/site-packages/keras/src/layers/core/embedding.py:97: UserWarning: Argument `input_length` is deprecated. Just remove it.
  warnings.warn(
E0000 00:00:1787605320.079831   13342 cuda_platform.cc:52] failed call to cuInit: INTERNAL: CUDA error: Failed call to cuInit: CUDA_ERROR_NO_DEVICE: no CUDA-capable device is detected


85/85 ━━━━━━━━━━━━━━━━━━━━ 3s 15ms/step - accuracy: 0.6885 - loss: 0.9863 - val_accuracy: 0.7500 - val_loss: 0.4853
Epoch 2/30
85/85 ━━━━━━━━━━━━━━━━━━━━ 1s 13ms/step - accuracy: 0.8174 - loss: 0.6306 - val_accuracy: 0.8294 - val_loss: 0.4264
Epoch 3/30
85/85 ━━━━━━━━━━━━━━━━━━━━ 1s 13ms/step - accuracy: 0.8848 - loss: 0.4446 - val_accuracy: 0.8235 - val_loss: 0.4868
Epoch 4/30
85/85 ━━━━━━━━━━━━━━━━━━━━ 1s 13ms/step - accuracy: 0.9234 - loss: 0.3149 - val_accuracy: 0.8118 - val_loss: 0.5766
Epoch 5/30
85/85 ━━━━━━━━━━━━━━━━━━━━ 1s 13ms/step - accuracy: 0.9411 - loss: 0.2314 - val_accuracy: 0.8059 - val_loss: 0.6462
11/11 ━━━━━━━━━━━━━━━━━━━━ 0s 11ms/step
              precision    recall  f1-score   support

    NonMatch     0.9180    0.8304    0.8720       283
       Match     0.4286    0.6316    0.5106        57

    accuracy                         0.7971       340
   macro avg     0.6733    0.7310    0.6913       340
weighted avg     0.8359    0.7971    0.8114       340


Fold 1
T

/home/ec2-user/anaconda3/envs/pytorch/lib/python3.10/site-packages/keras/src/layers/core/embedding.py:97: UserWarning: Argument `input_length` is deprecated. Just remove it.
  warnings.warn(


85/85 ━━━━━━━━━━━━━━━━━━━━ 3s 15ms/step - accuracy: 0.6764 - loss: 0.9654 - val_accuracy: 0.7971 - val_loss: 0.3855
Epoch 2/30
85/85 ━━━━━━━━━━━━━━━━━━━━ 1s 13ms/step - accuracy: 0.8155 - loss: 0.6462 - val_accuracy: 0.8206 - val_loss: 0.3809
Epoch 3/30
85/85 ━━━━━━━━━━━━━━━━━━━━ 1s 13ms/step - accuracy: 0.8877 - loss: 0.4523 - val_accuracy: 0.8235 - val_loss: 0.4077
Epoch 4/30
85/85 ━━━━━━━━━━━━━━━━━━━━ 1s 13ms/step - accuracy: 0.9197 - loss: 0.3200 - val_accuracy: 0.8206 - val_loss: 0.4937
Epoch 5/30
85/85 ━━━━━━━━━━━━━━━━━━━━ 1s 13ms/step - accuracy: 0.9462 - loss: 0.2180 - val_accuracy: 0.8088 - val_loss: 0.6099
11/11 ━━━━━━━━━━━━━━━━━━━━ 0s 11ms/step
              precision    recall  f1-score   support

    NonMatch     0.9421    0.8622    0.9004       283
       Match     0.5185    0.7368    0.6087        57

    accuracy                         0.8412       340
   macro avg     0.7303    0.7995    0.7545       340
weighted avg     0.8711    0.8412    0.8515       340


Fold 2
T

/home/ec2-user/anaconda3/envs/pytorch/lib/python3.10/site-packages/keras/src/layers/core/embedding.py:97: UserWarning: Argument `input_length` is deprecated. Just remove it.
  warnings.warn(


... (11 outputs left out)

/home/ec2-user/anaconda3/envs/pytorch/lib/python3.10/site-packages/keras/src/layers/core/embedding.py:97: UserWarning: Argument `input_length` is deprecated. Just remove it.
  warnings.warn(


85/85 ━━━━━━━━━━━━━━━━━━━━ 3s 16ms/step - accuracy: 0.6700 - loss: 0.9543 - val_accuracy: 0.6608 - val_loss: 0.6398
Epoch 2/30
85/85 ━━━━━━━━━━━━━━━━━━━━ 1s 13ms/step - accuracy: 0.8433 - loss: 0.5780 - val_accuracy: 0.7817 - val_loss: 0.5953
Epoch 3/30
85/85 ━━━━━━━━━━━━━━━━━━━━ 1s 13ms/step - accuracy: 0.9021 - loss: 0.3919 - val_accuracy: 0.7847 - val_loss: 0.6158
Epoch 4/30
85/85 ━━━━━━━━━━━━━━━━━━━━ 1s 13ms/step - accuracy: 0.9308 - loss: 0.2941 - val_accuracy: 0.7640 - val_loss: 0.7224
Epoch 5/30
85/85 ━━━━━━━━━━━━━━━━━━━━ 1s 13ms/step - accuracy: 0.9544 - loss: 0.1894 - val_accuracy: 0.8142 - val_loss: 0.7892
11/11 ━━━━━━━━━━━━━━━━━━━━ 0s 10ms/step
              precision    recall  f1-score   support

    NonMatch     0.9458    0.8021    0.8681       283
       Match     0.4343    0.7679    0.5548        56

    accuracy                         0.7965       339
   macro avg     0.6901    0.7850    0.7115       339
weighted avg     0.8613    0.7965    0.8163       339


Fold 9
T

/home/ec2-user/anaconda3/envs/pytorch/lib/python3.10/site-packages/keras/src/layers/core/embedding.py:97: UserWarning: Argument `input_length` is deprecated. Just remove it.
  warnings.warn(


85/85 ━━━━━━━━━━━━━━━━━━━━ 3s 15ms/step - accuracy: 0.7089 - loss: 0.9818 - val_accuracy: 0.5971 - val_loss: 0.6477
Epoch 2/30
85/85 ━━━━━━━━━━━━━━━━━━━━ 1s 13ms/step - accuracy: 0.8219 - loss: 0.6324 - val_accuracy: 0.7971 - val_loss: 0.4432
Epoch 3/30
85/85 ━━━━━━━━━━━━━━━━━━━━ 1s 13ms/step - accuracy: 0.8969 - loss: 0.4223 - val_accuracy: 0.8000 - val_loss: 0.5078
Epoch 4/30
85/85 ━━━━━━━━━━━━━━━━━━━━ 1s 13ms/step - accuracy: 0.9349 - loss: 0.2876 - val_accuracy: 0.8118 - val_loss: 0.5432
Epoch 5/30
85/85 ━━━━━━━━━━━━━━━━━━━━ 1s 13ms/step - accuracy: 0.9441 - loss: 0.2277 - val_accuracy: 0.8118 - val_loss: 0.6184
11/11 ━━━━━━━━━━━━━━━━━━━━ 0s 11ms/step
              precision    recall  f1-score   support

    NonMatch     0.9316    0.7703    0.8433       283
       Match     0.3810    0.7143    0.4969        56

    accuracy                         0.7611       339
   macro avg     0.6563    0.7423    0.6701       339
weighted avg     0.8407    0.7611    0.7861       339

      Mat

7. Save HAC-ready similarity scores

In [10]:
cnn_hac_score_df = cnn_pair_predictions_df[
    [
        "Model",
        "Fold",
        "InstanceID_1",
        "OrgName_1",
        "InstanceID_2",
        "OrgName_2",
        "TrueLabel",
        "PredictedLabel",
        "Probability_Match"
    ]
].copy()

# Rename match probability as similarity score for HAC clustering
cnn_hac_score_df = cnn_hac_score_df.rename(
    columns={"Probability_Match": "SimilarityScore"}
)

cnn_hac_score_df.to_csv(
    "cnn_hac_similarity_scores.csv",
    index=False,
    encoding="utf-8-sig"
)